# 1. Импорт библиотек и данных

In [2]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [9]:
train = pd.read_csv("../data/raw/train.csv")
X_train = train.drop('label', axis = 1).values.reshape(-1, 1, 28, 28).astype(np.float32) / 255.0
y_train = train['label'].values

In [11]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, train_size = 0.8, random_state = 42)

## 2. Создание тензоров для PyTorch

In [12]:
train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
valid_dataset = TensorDataset(torch.tensor(X_valid), torch.tensor(y_valid))

In [13]:
train_loader = DataLoader(train_dataset, batch_size = 64, shuffle = True)
valid_loader = DataLoader(valid_dataset, batch_size = 64)

## 3. Создание модели CNN

In [18]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size = 3, padding = 1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size = 3, padding = 1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

In [19]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.005)

In [20]:
EPOCHS = 10

In [25]:
for epoch in range(EPOCHS):
    model.train()
    for (images, labels) in train_loader:
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for (images, labels) in valid_loader:
            outputs = model(images)
            predicted = outputs.argmax(dim = 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"Epoch {epoch + 1}, Valid Accuracy: {100 * correct / total:.2f}%")

Epoch 1, Valid Accuracy: 98.82%
Epoch 2, Valid Accuracy: 98.82%
Epoch 3, Valid Accuracy: 98.77%
Epoch 4, Valid Accuracy: 98.68%
Epoch 5, Valid Accuracy: 98.56%
Epoch 6, Valid Accuracy: 98.85%
Epoch 7, Valid Accuracy: 98.64%
Epoch 8, Valid Accuracy: 98.98%
Epoch 9, Valid Accuracy: 98.93%
Epoch 10, Valid Accuracy: 98.80%


In [27]:
torch.save(model.state_dict(), 'mnist_cnn.pth')